<a href="https://colab.research.google.com/github/tuntunutycc/llamaindex/blob/main/L2_Tool_Calling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 2: Tool Calling

## Setup

In [ ]:
from helper import get_openai_api_key
OPENAI_API_KEY = get_openai_api_key()

In [ ]:
import nest_asyncio
nest_asyncio.apply()

## 1. Define a Simple Tool

FunctionTool ဟာ LlamaIndex framework ထဲက utility တစ်ခုဖြစ်ပြီး Python function တွေကို tool အဖြစ်ပြောင်းလဲဖန်တီးပေးတယ်။ ဒီ tool တွေကို LlamaIndex ရဲ့ agent တွေ၊ workflow တွေ၊ ဒါမှမဟုတ် query pipeline တွေမှာ အလိုအလျောက် လုပ်ဆောင်ချက်တွေ လုပ်ဖို့ အသုံးပြုနိုင်တယ်။

add function ရဲ့ docstring ("Adds two integers together.") က FunctionTool မှာ tool description အဖြစ် အလိုအလျောက်ထည့်ပေးပြီး agent တွေကို ဒီ tool က ဘာလုပ်တယ်ဆိုတာ သိစေတယ်။

FunctionTool.from_defaults(fn=add) ကို သုံးပြီး add function ကို LlamaIndex framework ထဲမှာ tool အဖြစ်ပြောင်းလဲဖန်တီးပေးပါတယ်။ အောက်မှာ အသေးစိတ်ရှင်းပြပါမယ်။




In [ ]:
from llama_index.core.tools import FunctionTool

def add(x: int, y: int) -> int:
    """Adds two integers together."""
    return x + y

def mystery(x: int, y: int) -> int:
    """Mystery function that operates on top of two numbers."""
    return (x + y) * (x + y)


add_tool = FunctionTool.from_defaults(fn=add)
mystery_tool = FunctionTool.from_defaults(fn=mystery)

llm.predict_and_call:
predict_and_call method က LLM (language model) ကို သုံးပြီး user query ကို process လုပ်တယ်။
Query အပေါ်မူတည်ပြီး ပေးထားတဲ့ tool တွေထဲက သင့်တော်တဲ့ tool ကို ရွေးပြီး ခေါ်သုံးတယ်။

In [ ]:
from llama_index.llms.openai import OpenAI

llm = OpenAI(model="gpt-3.5-turbo")
response = llm.predict_and_call(
    [add_tool, mystery_tool],
    "Tell me the output of the mystery function on 2 and 9",
    verbose=True
)
print(str(response))

## 2. Define an Auto-Retrieval Tool

### Load Data

To download this paper, below is the needed code:

#!wget "https://openreview.net/pdf?id=VtmBAGCN7o" -O metagpt.pdf

**Note**: The pdf file is included with this lesson. To access it, go to the `File` menu and select`Open...`.

In [ ]:
from llama_index.core import SimpleDirectoryReader
# load documents
documents = SimpleDirectoryReader(input_files=["metagpt.pdf"]).load_data()

In [ ]:
from llama_index.core.node_parser import SentenceSplitter
splitter = SentenceSplitter(chunk_size=1024)
nodes = splitter.get_nodes_from_documents(documents)

metadata_mode="all" parameter က metadata အားလုံးကိုပါ content နဲ့အတူ include လုပ်ဖို့ ညွှန်ကြားတယ်။

In [ ]:
print(nodes[0].get_content(metadata_mode="all"))

What is similarity_top_k?
similarity_top_k က vector-based search မှာ query နဲ့ အနီးစပ်ဆုံး (most similar) nodes ဒါမှမဟုတ် documents ဘယ်နှစ်ခုကို retrieve လုပ်မလဲဆိုတာကို ထိန်းချုပ်တဲ့ parameter ပါ။
LlamaIndex ရဲ့ VectorStoreIndex က documents တွေကို embeddings (numerical vectors) အဖြစ်ပြောင်းပြီး vector database ထဲမှာ သိမ်းထားတယ်။ Query လုပ်တဲ့အခါ user ရဲ့ input ကိုလည်း embedding အဖြစ်ပြောင်းပြီး vector space ထဲမှာ အနီးစပ်ဆုံး nodes တွေကို ရှာတယ်။
similarity_top_k=2 ဆိုတာက အနီးစပ်ဆုံး nodes/documents ၂ ခုကိုပဲ retrieve လုပ်မယ်လို့ ဆိုလိုတယ်။

In [ ]:
from llama_index.core import VectorStoreIndex

vector_index = VectorStoreIndex(nodes)
query_engine = vector_index.as_query_engine(similarity_top_k=2)

filters=MetadataFilters.from_dicts([...]):
Search results တွေကို metadata အပေါ်မူတည်ပြီး filter လုဪမယ်။
MetadataFilters.from_dicts က dictionary list ကနေ filter conditions တွေကို ဖန်တီးတယ်။
Filter condition: [{"key": "page_label", "value": "2"}]
ဒါက metadata ထဲမှာ "page_label" ဆိုတဲ့ key ရဲ့ value က "2" ဖြစ်တဲ့ nodes တွေကိုပဲ ယူမယ်လို့ ဆိုလိုတယ်။
ဥပမာ: Document တစ်ခုရဲ့ page number က 2 ဖြစ်တဲ့ nodes တွေကိုပဲ filter လုဪမယ်။

In [ ]:
from llama_index.core.vector_stores import MetadataFilters

query_engine = vector_index.as_query_engine(
    similarity_top_k=2,
    filters=MetadataFilters.from_dicts(
        [
            {"key": "page_label", "value": "2"}
        ]
    )
)

response = query_engine.query(
    "What are some high-level results of MetaGPT?",
)

In [ ]:
print(str(response))

In [ ]:
for n in response.source_nodes:
    print(n.metadata)

FilterCondition.OR ကို သုံးတဲ့အခါ metadata filters တွေထဲက တစ်ခုခုနဲ့ match ဖြစ်ရင် node ကို retrieve လုပ်မှာပါ။ ဆိုလိုတာက page_label က "1", "2", ဒါမှမဟုတ် "3" ထဲက တစ်ခုခုနဲ့ match ဖြစ်ရင် အဲဒီ node ကို search results ထဲမှာ ထည့်စဉ်းစားမယ်လို့ ဆိုလိုတယ်။

ဒါက broader search လိုချင်တဲ့အခါ (ဥပမာ: multiple page numbers တွေထဲက တစ်ခုခုကိုပဲ cover လုပ်ရင် ရတယ်) အသုံးဝင်ပါတယ်။

### Define the Auto-Retrieval Tool

In [ ]:
from typing import List
from llama_index.core.vector_stores import FilterCondition


def vector_query(
    query: str,
    page_numbers: List[str]
) -> str:
    """Perform a vector search over an index.

    query (str): the string query to be embedded.
    page_numbers (List[str]): Filter by set of pages. Leave BLANK if we want to perform a vector search
        over all pages. Otherwise, filter by the set of specified pages.

    """

    metadata_dicts = [
        {"key": "page_label", "value": p} for p in page_numbers
    ]

    query_engine = vector_index.as_query_engine(
        similarity_top_k=2,
        filters=MetadataFilters.from_dicts(
            metadata_dicts,
            condition=FilterCondition.OR
        )
    )
    response = query_engine.query(query)
    return response


vector_query_tool = FunctionTool.from_defaults(
    name="vector_tool",
    fn=vector_query
)

In [ ]:
llm = OpenAI(model="gpt-3.5-turbo", temperature=0)
response = llm.predict_and_call(
    [vector_query_tool],
    "What are the high-level results of MetaGPT as described on page 2?",
    verbose=True
)

In [ ]:
for n in response.source_nodes:
    print(n.metadata)

## Let's add some other tools!

In [ ]:
from llama_index.core import SummaryIndex
from llama_index.core.tools import QueryEngineTool

summary_index = SummaryIndex(nodes)
summary_query_engine = summary_index.as_query_engine(
    response_mode="tree_summarize",
    use_async=True,
)
summary_tool = QueryEngineTool.from_defaults(
    name="summary_tool",
    query_engine=summary_query_engine,
    description=(
        "Useful if you want to get a summary of MetaGPT"
    ),
)

In [ ]:
response = llm.predict_and_call(
    [vector_query_tool, summary_tool],
    "What are the MetaGPT comparisons with ChatDev described on page 8?",
    verbose=True
)

In [ ]:
for n in response.source_nodes:
    print(n.metadata)

In [ ]:
response = llm.predict_and_call(
    [vector_query_tool, summary_tool],
    "What is a summary of the paper?",
    verbose=True
)